# Mean–Variance Portfolio Theory

Mean–variance analysis identifies how investments can be combined to balance expected return and risk, connecting portfolio construction to asset pricing.

Under CAPM's assumptions, the market is the tangency portfolio: no other combination offers a higher expected excess return per unit of volatility. However, a strategy with nonzero population alpha challenges that proposition. After hedging its market exposure, it leaves an expected payoff outside the market benchmark. Under unrestricted trading, combining that payoff with the market expands the investment opportunity set and increases the maximum attainable Sharpe ratio.

In this notebook, I explain the connection through portfolio return and risk, the efficient frontier, and the tangency portfolio. I then use the supplied factor returns to compare the MKT–SMB–HML subset with all five factors, showing how their estimated frontiers, tangency combinations, and exposures differ. The application describes historical opportunities estimated in sample, rather than future returns an investor is assured of earning.

In [ ]:
import sys
from pathlib import Path
ROOT=Path.cwd().resolve()
ROOT=ROOT.parent if ROOT.name=='notebooks' else ROOT
sys.path.insert(0,str(ROOT/'src'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
NAVY, BLUE, GREEN, ORANGE, GRAY = '#17324D', '#2878A5', '#168A86', '#D99A32', '#737D87'
STYLE={'font.family':'DejaVu Sans','font.size':11,'axes.titlesize':12,'axes.labelsize':11,
       'xtick.labelsize':10,'ytick.labelsize':10,'legend.fontsize':10,'figure.dpi':110,
       'figure.facecolor':'white','axes.facecolor':'white','axes.spines.top':False,
       'axes.spines.right':False,'axes.prop_cycle':plt.cycler(color=[BLUE,GREEN,ORANGE,NAVY]),
       'grid.alpha':.15,'lines.linewidth':1.8}
plt.rcParams.update(STYLE)
from factor_asset_pricing.mean_variance import global_minimum_variance, tangency_portfolio, efficient_frontier, portfolio_statistics
from factor_asset_pricing.asset_pricing_tests import maximum_sharpe_ratio
from factor_asset_pricing.french_data import load_french, sample_coverage
RAW = ROOT / "data/raw/french"


## 1. Mean–variance portfolio theory

### 1.1 Portfolio return, risk, and diversification

We usually hold investments together, not in isolation. What an investment contributes therefore depends on both its expected return and how it moves with the rest of the portfolio. For example, a modest standalone Sharpe ratio does not rule out a useful contribution through diversification.

Portfolio expected return is the weighted average of the individual expected returns. Portfolio risk also depends on how those returns move together:

$$\mu_p=\mathbf w'\boldsymbol\mu,\qquad\sigma_p^2=\mathbf w'\Sigma\mathbf w.$$

Here $\mathbf w$ contains the portfolio weights, $\boldsymbol\mu$ the expected returns, and $\Sigma$ the return covariance matrix. The resulting portfolio expected return is $\mu_p$, and its variance is $\sigma_p^2$.

For two investments, the risk calculation becomes

$$\sigma_p^2=w_1^2\sigma_1^2+w_2^2\sigma_2^2+2w_1w_2\operatorname{Cov}(R_1,R_2).$$

For positive positions, lower covariance reduces combined variance without changing either investment's standalone volatility. Negative positions can hedge common fluctuations. Their use also changes leverage and implementation requirements.

For ordinary fully invested risky assets, $\mathbf1'\mathbf w=1$ is a capital-budget constraint. For constructed long–short factor payoffs, the same moment formulas apply to exposure notionals, but a unit-sum condition is a chosen normalization rather than an intrinsic capital budget. We distinguish these interpretations in the example.

### 1.2 The efficient frontier

Mean–variance optimization asks which combination has the lowest variance for a chosen expected return (Markowitz, 1952):

$$
\min_{\mathbf w}\ \mathbf w'\Sigma\mathbf w
\quad\text{subject to}\quad\mathbf1'\mathbf w=1,\quad\mathbf w'\boldsymbol\mu=\mu^*.
$$

Here $\mu^*$ is the target expected return. The weights sum to one and can be negative, so the solution can include short positions. The optimal weights and resulting minimum variance are:

$$
\mathbf w^*(\mu^*)=\frac{C-B\mu^*}{D}\Sigma^{-1}\mathbf1+\frac{A\mu^*-B}{D}\Sigma^{-1}\boldsymbol\mu.\qquad\sigma_p^2(\mu^*)=\frac{A(\mu^*)^2-2B\mu^*+C}{D}.
$$

where

$$
A=\mathbf1'\Sigma^{-1}\mathbf1,\qquad B=\mathbf1'\Sigma^{-1}\boldsymbol\mu,\qquad C=\boldsymbol\mu'\Sigma^{-1}\boldsymbol\mu,\qquad D=AC-B^2.
$$

To draw the curve, we choose a range of target returns, substitute each target into the weight formula, and calculate the corresponding volatility. Expected return goes on the vertical axis and the square root of variance on the horizontal axis. The library's `efficient_frontier` function uses exactly this analytical weight formula. It also calculates variance from the resulting weights, which gives the same value as the expression above.

Together, the points trace the minimum-variance boundary. The upper branch is the efficient frontier. On the lower branch, another portfolio offers a higher expected return for the same volatility. These are estimated opportunities under the stated trading assumptions. Different expected-return estimates or restrictions on short selling can change the curve.

### 1.3 The minimum-variance portfolio

The global minimum-variance portfolio, or GMV, is the lowest-risk combination under the unit-sum constraint, without imposing a target expected return. Its weights are

$$
\mathbf w_{GMV}=\frac{\Sigma^{-1}\mathbf1}{\mathbf1'\Sigma^{-1}\mathbf1}.
$$

Only the covariance matrix enters this calculation. GMV does not use expected returns to choose its weights. The library calculates it with `global_minimum_variance`.

GMV is the leftmost point of the minimum-variance boundary. Moving along the upper branch means taking more volatility in exchange for a higher expected return. The lowest-risk portfolio is therefore not automatically the one with the best Sharpe ratio or the one every investor should hold.


### 1.4 The tangency portfolio

With a risk-free asset available, the tangency portfolio is the risky combination with the highest expected excess return per unit of volatility. Tobin's (1958) separation argument distinguishes choosing this risky combination from deciding how much to invest in it.

We start with expected returns, the covariance matrix, and a risk-free return for the same period. Here $\boldsymbol\mu$ contains expected total returns, $r_f$ is the risk-free return, and $\mathbf1$ is a vector of ones. The Sharpe ratio is

$$SR(\mathbf w)=\frac{\mathbf w'(\boldsymbol\mu-r_f\mathbf1)}{\sqrt{\mathbf w'\Sigma\mathbf w}}.$$

The slope is measured from the risk-free return. The numerator is expected return above that alternative, and the denominator is portfolio volatility. We therefore want the combination with the steepest trade-off between expected excess return and risk. With unrestricted positions, the tangency weights are

$$\mathbf w_T=\frac{\Sigma^{-1}(\boldsymbol\mu-r_f\mathbf1)}{\mathbf1'\Sigma^{-1}(\boldsymbol\mu-r_f\mathbf1)}.$$

To read this formula, first subtract the risk-free return from each expected return. The inverse covariance matrix then accounts for how the investments move together. Finally, divide by the sum to obtain weights that add to one. The library's `tangency_portfolio` function performs this calculation.

Combining the tangency portfolio with the risk-free asset gives the capital allocation line. In total-return space, it starts at $(0,r_f)$ and its slope is the Sharpe ratio. Holding some wealth risk-free reduces exposure to the risky combination. Borrowing at the same risk-free rate increases it. Under these idealized assumptions, positive scaling changes expected excess return and volatility proportionally, leaving the Sharpe ratio unchanged.

For the factor payoffs used below, $\mathbf F_t$ already contains excess returns. We therefore use $E[\mathbf F_t]$ directly in place of $\boldsymbol\mu-r_f\mathbf1$, without subtracting RF again. The allocation lines start at the origin in excess-payoff space. Unit-sum factor weights describe normalized exposures, rather than the net cash invested in the underlying long and short positions.


## 2. Example: from three-factor to five-factor investment opportunities

### 2.1 Factor returns and the exposure convention

I use published FF5 returns from the [Kenneth French Data Library](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html) on the same complete-case factor sample as 05: July 1963 through December 2024, totaling 738 consecutive months. The comparison asks what changes when we give the investor more factors to combine. MKT is the market-only benchmark. The three-factor subset contains MKT, SMB, and HML, while FF5 additionally contains RMW and CMA.

The supplied SMB follows the five-factor construction explained in 03. The three-factor subset is therefore a nested benchmark using that SMB, rather than an exact reconstruction of conventional FF3. All inputs are already excess-return payoffs. Sample means are arithmetic monthly averages, and covariance uses divisor $T-1$.

To compare the curved frontiers, I impose $\mathbf1'\mathbf w=1$ on signed factor exposures. This defines comparable unit-sum exposure combinations for both investment sets. It does not assert that a dollar-neutral factor requires one dollar of net investment, or that these exposures form an ordinary fully invested risky portfolio. GMV refers to minimum variance under this specific normalization. Without a scale constraint, zero exposure would trivially minimize factor-payoff variance.

The maximum-Sharpe direction does not depend on positive scaling. The figure shows both the unrestricted allocation line and the unit-sum tangency point, provided the normalization preserves its orientation. Both investment sets satisfy that condition in this sample.

In [ ]:
factors = load_french("ff5", RAW).loc["1963-07":"2024-12"]
assert len(factors) == 738 and not factors.isna().any().any()
factor_panel=factors[["MKT","SMB","HML","RMW","CMA"]]
print(f"Common monthly sample: {factor_panel.index.min():%B %Y}–{factor_panel.index.max():%B %Y}, T = {len(factor_panel)}")
inputs=pd.DataFrame({"Mean (% monthly)":factor_panel.mean()*100,
                     "Volatility (% monthly)":factor_panel.std(ddof=1)*100,
                     "Monthly Sharpe":factor_panel.mean()/factor_panel.std(ddof=1)})
display(inputs)
correlations=factor_panel.corr()
correlations.index.name="Factor correlations"
display(correlations)
display(sample_coverage(factor_panel))


Looking at each factor separately is not enough to choose the best combination. The correlation table helps explain why: HML and CMA are positively correlated at approximately 0.69, while SMB and RMW have a correlation of approximately −0.35. Several characteristic factors also have negative correlations with MKT. Their contributions depend on these joint relationships, not only on their individual Sharpe ratios.

### 2.2 Estimated frontiers and tangency portfolios

I calculate the frontier, GMV, and tangency combinations using the same estimated means and covariance matrix. For the already-excess factor payoffs, the zero intercept of the allocation lines represents zero excess payoff, not a second risk-free deduction. Solid curved lines show efficient branches and faint dashed curves show dominated branches.

The two investment sets are nested. An existing three-factor allocation remains feasible in FF5 by setting RMW and CMA exposures to zero. Thus the added factors cannot increase the minimum variance at any shared target mean under the same unit-sum constraint.

In [ ]:
sets={"Three-factor subset":["MKT","SMB","HML"],"FF5":["MKT","SMB","HML","RMW","CMA"]}
allocations={}
performance_rows={}
weight_columns={}
for name,columns in sets.items():
    returns=factor_panel[columns]
    mean=returns.mean().to_numpy()
    covariance=returns.cov().to_numpy()
    gmv=global_minimum_variance(covariance)
    tangency=tangency_portfolio(mean,covariance,risk_free_rate=0)
    stats={kind:portfolio_statistics(w,mean,covariance,risk_free_rate=0)
           for kind,w in [("GMV",gmv),("Tangency",tangency)]}
    allocations[name]={"columns":columns,"mean":mean,"covariance":covariance,
                       "GMV":gmv,"Tangency":tangency,"statistics":stats}
    for kind,w in [("GMV",gmv),("Tangency",tangency)]:
        label=f"{name}: {kind}"
        s=stats[kind]
        performance_rows[label]={"Mean (% monthly)":s["expected_return"]*100,
                                 "Volatility (% monthly)":s["volatility"]*100,
                                 "Monthly Sharpe":s["sharpe"]}
        weight_columns[label]=pd.Series(w,index=columns).reindex(factor_panel.columns,fill_value=0)
market_stats=portfolio_statistics([1],factor_panel[["MKT"]].mean(),factor_panel[["MKT"]].cov(),risk_free_rate=0)
performance_rows={"MKT only":{"Mean (% monthly)":market_stats["expected_return"]*100,
                              "Volatility (% monthly)":market_stats["volatility"]*100,
                              "Monthly Sharpe":market_stats["sharpe"]},**performance_rows}
performance=pd.DataFrame(performance_rows).T
weights=pd.DataFrame(weight_columns)
targets=np.linspace(0,.009,241)
for name,a in allocations.items():
    a["frontier"]=efficient_frontier(a["mean"],a["covariance"],target_returns=targets)

In [ ]:
fig,ax=plt.subplots(figsize=(9,6),layout="constrained")
colors={"Three-factor subset":BLUE,"FF5":GREEN}
volatility_grid=np.linspace(0,.06,200)
ax.plot(volatility_grid*100,volatility_grid*market_stats["sharpe"]*100,color=GRAY,ls=":",label="MKT allocation line")
for name,a in allocations.items():
    color=colors[name]
    frontier=a["frontier"]
    gmv=a["statistics"]["GMV"]
    tangent=a["statistics"]["Tangency"]
    upper=frontier["expected_return"]>=gmv["expected_return"]
    ax.plot(frontier.loc[upper,"volatility"]*100,frontier.loc[upper,"expected_return"]*100,color=color,label=f"{name}: efficient frontier")
    ax.plot(frontier.loc[~upper,"volatility"]*100,frontier.loc[~upper,"expected_return"]*100,color=color,ls="--",alpha=.4,lw=1)
    ax.plot(volatility_grid*100,volatility_grid*tangent["sharpe"]*100,color=color,ls=":",label=f"{name}: maximum-Sharpe line")
    ax.scatter(gmv["volatility"]*100,gmv["expected_return"]*100,color=color,marker="o",s=45,zorder=4,label=f"{name}: GMV")
    ax.scatter(tangent["volatility"]*100,tangent["expected_return"]*100,color=color,marker="*",s=160,zorder=5,label=f"{name}: tangency")
ax.scatter(market_stats["volatility"]*100,market_stats["expected_return"]*100,color=GRAY,marker="s",s=50,zorder=4,label="MKT only")
ax.set(xlim=(0,6),ylim=(0,1.1),xlabel="Monthly payoff volatility (%)",ylabel="Mean excess payoff (% per month)",title="Estimated factor frontiers and maximum-Sharpe lines")
ax.legend(fontsize=8,ncol=2,loc="upper center",bbox_to_anchor=(.5,-.14))
plt.show()

The FF5 efficient frontier lies to the left of the three-factor frontier at a shared target mean, indicating less estimated variance for that mean under the stated constraint. Its maximum-Sharpe line is also steeper. That means more estimated excess return per unit of volatility. GMV and tangency still answer different questions, even though the FF5 points lie close together in this sample.

The market-only monthly Sharpe ratio is approximately 0.131. The maximum is approximately 0.182 for the three-factor subset and 0.295 for FF5. These are in-sample estimates from identical dates. They show that the additional factors improve the estimated combinations, without proving that the population market or three-factor benchmark is inefficient. The CAPM-spanning results in 05 use MKT alone and do not test the incremental role of RMW and CMA against the three-factor subset.

### 2.3 Exposures and the resulting portfolios

The tables show which factor positions sit behind the points in the figure, together with their expected payoffs and volatility. Exposures sum to one in every displayed combination, but can be negative. A coefficient of 0.4 scales the reported factor payoff by 0.4. It does not mean that 40% of net cash wealth has been invested in a dollar-neutral factor. Underlying long and short positions, financing, collateral, and gross exposure require a separate implementation analysis.

In [ ]:
weights.index.name="Factor exposure"
display(weights)
display(performance)
fig,ax=plt.subplots(figsize=(8.5,4.6),layout="constrained")
positions=np.arange(len(factor_panel.columns))
width=.34
for offset,(name,color) in enumerate(colors.items()):
    column=f"{name}: Tangency"
    ax.bar(positions+(offset-.5)*width,weights[column],width=width,color=color,label=name)
ax.axhline(0,color=NAVY,lw=1)
ax.set(xticks=positions,xticklabels=factor_panel.columns,xlabel="Factor",ylabel="Signed factor exposure (unit-sum normalization)",title="Tangency combinations: factor exposures")
ax.legend(fontsize=9)
plt.show()

The three-factor tangency combination assigns approximately 40.7% exposure to MKT, 8.3% to SMB, and 50.9% to HML. The FF5 combination instead assigns approximately 17.7% to MKT, 11.6% to SMB, −5.3% to HML, 34.0% to RMW, and 42.0% to CMA.

The negative HML coefficient means taking the opposite side of the usual HML position, reversing its long and short legs. It is a covariance-adjusted hedge within the estimated combination, rather than a conclusion that value investing has a negative expected return. To understand that position, we need to look at the positive HML–CMA correlation and the other exposures together. The FF5 signed exposures sum to one, while their absolute values sum to approximately 1.11. That gross factor-exposure measure is not the same as the underlying securities' gross exposure.

At the chosen normalization, FF5 tangency has an estimated mean excess payoff of approximately 0.313% per month and volatility of 1.061%. Three-factor tangency has a higher mean, approximately 0.398%, but volatility of 2.195%. So the larger FF5 Sharpe ratio does not come from a higher mean alone. It comes from how the expected payoffs and covariances work together. Positive rescaling could place either combination at another volatility without changing its Sharpe ratio.

### 2.4 Ruling out negative factor weights

An investor may prefer to avoid short positions, or face restrictions that prevent them. I therefore repeat the FF5 example with nonnegative factor weights. This means we can hold a factor or leave it out, but cannot take the opposite side of it. The same restriction applies to both the minimum-variance and tangency portfolios.

For the frontier, we again find the lowest variance at each target return, now with the additional restriction:

$$
\min_{\mathbf w}\ \mathbf w'\Sigma\mathbf w
\quad\text{subject to}\quad\mathbf1'\mathbf w=1,\quad\mathbf w'\boldsymbol\mu=\mu^*,\quad w_i\geq0.
$$

The additional constraint reduces the combinations available to us. For each target return, I use a numerical optimizer to find the lowest-variance combination that satisfies all three conditions. The earlier analytical formula allows negative weights, so it no longer solves this restricted problem. Simply replacing its negative weights with zero would not give the optimal combination either.

For the minimum-variance portfolio, I remove the target-return condition and find the lowest-risk combination with nonnegative weights. For the tangency portfolio, I instead choose the nonnegative weights with the highest Sharpe ratio. The tables compare both portfolios with their unrestricted counterparts.

Here “no shorting” refers to the **factor weights**. HML and the other long–short factors still contain short stock positions, even when their factor weights are positive.


In [ ]:
from scipy.optimize import minimize

a = allocations["FF5"]
mean, covariance = a["mean"], a["covariance"]
n_factors = len(mean)
initial = np.full(n_factors, 1 / n_factors)
bounds = [(0, 1)] * n_factors
budget = {"type": "eq", "fun": lambda w: w.sum() - 1}
# Scaling the variance objective improves numerical accuracy without changing the optimum.
variance_scale = np.trace(covariance)

def restricted_weights(objective, target=None):
    constraints = [budget]
    if target is not None:
        constraints.append({"type": "eq", "fun": lambda w: (w @ mean - target) * 100})
    result = minimize(objective, initial, method="SLSQP", bounds=bounds,
                      constraints=constraints, options={"ftol": 1e-12, "maxiter": 1000})
    if not result.success:
        raise RuntimeError(result.message)
    w = result.x
    assert np.isclose(w.sum(), 1, atol=1e-8) and w.min() >= -1e-8
    if target is not None:
        assert np.isclose(w @ mean, target, atol=1e-8)
    return w

variance_objective = lambda w: (w @ covariance @ w) / variance_scale
restricted_gmv = restricted_weights(variance_objective)
restricted_tangency = restricted_weights(
    lambda w: -(w @ mean) / np.sqrt(w @ covariance @ w))
restricted_targets = np.linspace(restricted_gmv @ mean, mean.max(), 100)
restricted_frontier = []
for target in restricted_targets:
    w = restricted_weights(variance_objective, target)
    unrestricted = efficient_frontier(mean, covariance, target_returns=[target]).iloc[0]
    stats = portfolio_statistics(w, mean, covariance, risk_free_rate=0)
    assert stats["volatility"] ** 2 >= unrestricted["volatility"] ** 2 - 1e-9
    restricted_frontier.append(stats)
restricted_frontier = pd.DataFrame(restricted_frontier)

comparison_weights = pd.DataFrame({
    "Unrestricted GMV": a["GMV"], "Nonnegative GMV": restricted_gmv,
    "Unrestricted tangency": a["Tangency"], "Nonnegative tangency": restricted_tangency,
}, index=a["columns"])
comparison_stats = pd.DataFrame({
    label: portfolio_statistics(w, mean, covariance, risk_free_rate=0)
    for label, w in comparison_weights.items()
}).T
assert comparison_stats.loc["Nonnegative tangency", "sharpe"] <= a["statistics"]["Tangency"]["sharpe"] + 1e-8
display(comparison_weights)
display(comparison_stats.rename(columns={
    "expected_return": "Mean (% monthly)", "volatility": "Volatility (% monthly)",
    "sharpe": "Monthly Sharpe"})[["Mean (% monthly)", "Volatility (% monthly)", "Monthly Sharpe"]]
    .mul([100, 100, 1]))

fig, ax = plt.subplots(figsize=(8, 5), layout="constrained")
shared_targets = np.linspace(a["statistics"]["GMV"]["expected_return"], mean.max(), 150)
unrestricted_frontier = efficient_frontier(mean, covariance, target_returns=shared_targets)
ax.plot(unrestricted_frontier["volatility"] * 100,
        unrestricted_frontier["expected_return"] * 100, color=GREEN, label="Unrestricted FF5")
ax.plot(restricted_frontier["volatility"] * 100,
        restricted_frontier["expected_return"] * 100, color=ORANGE, label="Nonnegative factor weights")
for label, color in [("Unrestricted tangency", GREEN), ("Nonnegative tangency", ORANGE)]:
    s = comparison_stats.loc[label]
    ax.scatter(s["volatility"] * 100, s["expected_return"] * 100,
               marker="*", s=150, color=color, zorder=4)
ax.set(xlabel="Monthly payoff volatility (%)", ylabel="Mean excess payoff (% per month)",
       title="FF5 efficient frontiers with and without negative factor weights")
ax.legend()
plt.show()


The restricted frontier cannot move to the left of the unrestricted frontier at the same expected payoff. We have removed possible combinations without adding any new ones. Both restricted portfolios leave HML out and adjust their other weights. The minimum-variance portfolio's monthly volatility rises slightly, from approximately 1.014% to 1.029%. Avoiding negative factor weights therefore gives up a small amount of the risk reduction available in the unrestricted combination.

The stars mark the two maximum-Sharpe combinations. With nonnegative weights, HML receives zero exposure and the other weights adjust. The monthly Sharpe ratio falls only slightly, from approximately 0.295 to 0.293. In this sample, ruling out negative factor weights therefore costs little in terms of the best estimated return-to-risk trade-off. These remain in-sample estimates based on the same historical means and covariances.


### 2.5 What the estimated expansion establishes

This example shows what an improvement in investment opportunities looks like on the frontier. Adding RMW and CMA preserves the original feasible allocations and, in this sample, supplies combinations with lower variance at a target payoff and higher maximum Sharpe. The displayed exposures identify the combinations behind that improvement.

All means, covariances, and positions are estimated using the full historical sample. Selecting the combination with the best Sharpe on that same history makes the reported optimum optimistic as an assessment of future performance. Small changes in mean estimates or covariance can also materially change exposures, especially when large offsetting positions are permitted.

Apart from the nonnegative-factor-weight example, the calculations allow signed exposures. They impose no transaction costs, financing limits, or collateral requirements. They describe estimated payoff opportunities under idealized trading assumptions, rather than an implemented strategy. Out-of-sample evidence would be needed to evaluate whether an investor could realize the improvement.

CAPM's market-efficiency claim and the empirical factor combination thus remain distinct. Notebook 08 evaluates joint alpha restrictions for test portfolios and relates that evidence to mean–variance efficiency under the relevant assumptions.

## References

- Markowitz, H. (1952). Portfolio selection. *The Journal of Finance*, 7(1), 77–91. [DOI](https://doi.org/10.1111/j.1540-6261.1952.tb01525.x).
- Tobin, J. (1958). Liquidity preference as behavior towards risk. *The Review of Economic Studies*, 25(2), 65–86. [DOI](https://doi.org/10.2307/2296205).
- Kenneth R. French Data Library. [Factor and portfolio data](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html).
